In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import plotly.express as px
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate


from utilsforecast.losses import *

from utilsforecast.losses import *

import plotly.io as pio

from utilsforecast.losses import *
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial
from sklearn.linear_model import RidgeCV
from utilsforecast.losses import *
from plotting_utils import (
    plot_data_availability_heatmap,
    plot_missing_percentage,
    plotly_series as plot_series,
)
from statsforecast.models import SklearnModel

from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    MSTL,
)
from xgboost import XGBRegressor

# Introduction to Probabilistic Forecasting

Traditional time series forecasting methods typically provide a single predicted value for each future time point. This is known as **point forecasting**. However, real-world data is often uncertain and subject to various sources of randomness. For example, predicting tomorrow's electricity consumption or next week's sales involves many unknown factors.

**Probabilistic forecasting** addresses this uncertainty by predicting a range of possible future values, along with their associated probabilities. Instead of answering "What is the most likely value?", probabilistic forecasting answers "What is the probability that the value will fall within a certain range?".

## Why Probabilistic Forecasts Matter

- **Quantifying Uncertainty:** Probabilistic forecasts provide a measure of confidence in predictions, which is crucial for risk management and decision-making.
- **Better Decision Support:** Businesses can plan for best-case, worst-case, and most-likely scenarios.
- **Real-World Relevance:** Many applications (e.g., energy demand, finance, weather) require understanding the full range of possible outcomes, not just the average.

## Key Concepts

- **Prediction Interval:** A range within which the future value is expected to fall with a certain probability (e.g., 95% prediction interval).
- **Forecast Distribution:** The full probability distribution of possible future values, not just a single point estimate.

For example, instead of predicting that tomorrow's energy consumption will be exactly 100 kWh, a probabilistic forecast might say:

> There is a 90% chance that tomorrow's energy consumption will be between 95 and 110 kWh.

Mathematically, if $y_{t+h}$ is the value we want to forecast at time $t+h$, a probabilistic forecast provides the conditional distribution $P(y_{t+h} \mid \text{past data})$.

In the next sections, we'll explore how to generate and interpret probabilistic forecasts using modern time series tools.

In [3]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [4]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [5]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [6]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


# Conformal Prediction Intervals: A Modern Approach to Reliable Uncertainty Quantification

## What Are Conformal Prediction Intervals?

Conformal prediction is a cutting-edge framework for constructing **prediction intervals** (or sets) that are valid under minimal assumptions. Unlike traditional statistical models—which often rely on strong distributional assumptions (like normality or independence)—conformal prediction provides **finite-sample, distribution-free guarantees**. This means that, no matter the underlying data distribution, the intervals will contain the true value with a user-specified probability (e.g., 90% or 95%), as long as the data are **exchangeable** (a slightly weaker condition than independence).

### Why Are Conformal Prediction Intervals Important?

- **Distribution-Free:** No need to assume normality, constant variance, or any specific error distribution.
- **Finite-Sample Guarantees:** The coverage probability (e.g., 90%) holds even for small datasets, not just asymptotically.
- **Model-Agnostic:** Can be applied on top of any forecasting model—statistical, machine learning, or deep learning.
- **Robustness:** Particularly useful when traditional model assumptions are questionable or when data are complex and non-standard.

## How Does Conformal Prediction Work? (Step-by-Step)

Let's break down the conformal prediction process for time series forecasting into clear, beginner-friendly steps:

### 1. **Train a Forecasting Model**

First, fit your favorite forecasting model (e.g., ARIMA, XGBoost, neural network) on your training data. This model will provide **point forecasts** for future values.

### 2. **Compute Nonconformity Scores (Residuals)**

For each observation in a **calibration set** (a hold-out set not used for training), calculate a **nonconformity score**. The most common choice is the **absolute residual**:

$$
\text{Nonconformity score for time } t = |y_t - \hat{y}_t|
$$

- $y_t$: The true observed value at time $t$.
- $\hat{y}_t$: The model's prediction at time $t$.

### 3. **Determine the Quantile Threshold**

Decide on your desired coverage level, say $1 - \alpha = 0.9$ for a 90% interval. Find the $(1 - \alpha)$-quantile of the nonconformity scores from the calibration set:

$$
q = \text{Quantile}_{1-\alpha}(\{\text{nonconformity scores}\})
$$

This $q$ is the "width" you need to add to your point forecast to achieve the desired coverage.

### 4. **Construct the Prediction Interval**

For a new forecast $\hat{y}_{t+h}$, the conformal prediction interval is:

$$
[\hat{y}_{t+h} - q, \; \hat{y}_{t+h} + q]
$$

This interval is guaranteed to contain the true value $y_{t+h}$ with probability at least $1 - \alpha$, under the exchangeability assumption.

---

## Real-World Analogy

Imagine you’re a weather forecaster. You make predictions for tomorrow’s temperature, but you know your model isn’t perfect. By looking at how far off your past predictions were (the residuals), you can estimate how much "wiggle room" to add to your next forecast. Conformal prediction formalizes this idea, ensuring that your intervals are neither too narrow nor too wide, and that they work reliably—even if the weather behaves unpredictably!

---

## Mathematical Summary

Given a set of calibration residuals $\{r_1, r_2, ..., r_n\}$, the conformal prediction interval for a new forecast $\hat{y}_{\text{new}}$ at coverage $1-\alpha$ is:

$$
\left[ \hat{y}_{\text{new}} - q, \; \hat{y}_{\text{new}} + q \right]
$$

where $q$ is the $(1-\alpha)$-quantile of the calibration residuals.

---

## Key Properties and Advantages

- **Valid Coverage:** The interval contains the true value with at least the specified probability, regardless of the underlying data distribution.
- **Flexible:** Works with any regression or forecasting model.
- **Simple to Implement:** Only requires residuals from a calibration set and quantile computation.

---

## Extensions and Variants

- **Asymmetric Intervals:** If residuals are not symmetric, you can use separate quantiles for the lower and upper bounds.
- **Adaptive/Local Conformal:** Adjust the interval width based on local model uncertainty or covariates.
- **Time Series Considerations:** For time series, special care is needed to avoid "peeking into the future"—the calibration set should always precede the test set in time.

---

## Example: Conformal Prediction in Time Series Forecasting

Suppose you have a time series of electricity consumption and you want to forecast the next 48 half-hour periods. Here’s how you might use conformal prediction:

1. **Train your model** on data up to time $T-48$.
2. **Use the next 48 points** ($T-48$ to $T$) as your calibration set. Compute residuals $r_t = |y_t - \hat{y}_t|$.
3. **For each future forecast** $\hat{y}_{T+h}$ ($h=1$ to $48$), construct the interval:

    $$
    [\hat{y}_{T+h} - q, \; \hat{y}_{T+h} + q]
    $$

    where $q$ is the 90th percentile of the calibration residuals for a 90% interval.

---

## Practical Tips for Beginners

- **Always use a separate calibration set** (not used for model training) to compute residuals.
- **Intervals may be wider** than those from traditional models, especially if your model is underfitting or the data are noisy. This is a feature, not a bug—it reflects true uncertainty!
- **Conformal prediction is especially useful** when you don’t trust the assumptions of your model, or when using complex ML models that don’t provide natural prediction intervals.

---

## Limitations and Considerations

- **Exchangeability Assumption:** The method assumes that the calibration and test data are exchangeable (i.e., drawn from the same distribution). In time series, this can be tricky if the data are non-stationary.
- **Interval Width:** If your model is poor, the intervals will be wide—conformal prediction cannot "fix" a bad model, but it will honestly reflect its uncertainty.
- **Computational Cost:** For some advanced conformal methods (like those for quantile regression or time series with covariates), implementation can be more complex.

---

## Summary Table: Conformal vs. Classical Prediction Intervals

| Feature                | Classical (e.g., ARIMA/ETS) | Conformal Prediction      |
|------------------------|-----------------------------|--------------------------|
| Assumptions            | Often normality, homoscedasticity | Exchangeability only     |
| Model dependency       | Model-specific formulas      | Model-agnostic           |
| Coverage guarantee     | Asymptotic, model-dependent | Finite-sample, distribution-free |
| Flexibility            | Limited                     | Very high                |
| Handles ML models      | No                          | Yes                      |

---

## Further Reading

- **Tutorial:** [A Gentle Introduction to Conformal Prediction and Its Applications in Machine Learning](https://arxiv.org/abs/2107.07511)
- **Time Series Application:** [Conformalized Quantile Regression for Time Series](https://arxiv.org/abs/1905.05301)
- **Nixtla’s Implementation:** The [Nixtla](https://nixtla.github.io/) libraries support conformal prediction for time series forecasting, making it easy to add reliable uncertainty estimates to your forecasts.

---

**In summary:**  
Conformal prediction intervals are a powerful, modern tool for quantifying uncertainty in time series forecasting. They provide robust, honest, and model-agnostic intervals that work with any forecasting method—making them an essential addition to the toolkit of any data scientist or time series analyst.

In [ ]:
# https://github.com/Nixtla/statsforecast/blob/main/python/statsforecast/models.py#L152
# https://github.com/sktime/sktime/blob/v0.38.1/sktime/forecasting/conformal.py#L470-L477

In [8]:
from mlforecast.lag_transforms import (
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [9]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data.select([id_, time_, target_]),
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)

mlf = MLForecast(
    models=[],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

In [ ]:
from statsforecast.utils import ConformalIntervals


In [33]:
# Create a list of models and instantiation parameters
intervals = ConformalIntervals(h=48, n_windows=100)
# P.S. n_windows*h should be less than the count of data elements in your time series sequence.
# P.S. Also value of n_windows should be atleast 2 or more.

level = [80]
models = [
    SklearnModel(
        XGBRegressor(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=42,
            verbosity=0,
        ),
        prediction_intervals=intervals,
    ),
]

sf = StatsForecast(
    models=models,
    freq="30m",
)

y_hat = (
    sf.cross_validation(
        df=mlf.preprocess(data_fourier, static_features=[]),
        h=48,
        step_size=1,
        n_windows=1,
        level=level,
    )
    .drop("cutoff")
    .with_columns(pl.col("XGBRegressor-lo-80").clip(lower_bound=0))
)

In [34]:
y_hat

unique_id,ds,y,XGBRegressor,XGBRegressor-lo-80,XGBRegressor-hi-80
str,datetime[ns],f64,f64,f64,f64
"""MAC000193""",2014-02-27 00:00:00,0.953,0.291296,0.0,0.605008
"""MAC000193""",2014-02-27 00:30:00,0.012,0.37566,0.182781,0.568539
"""MAC000193""",2014-02-27 01:00:00,0.033,0.014968,0.0,0.05894
"""MAC000193""",2014-02-27 01:30:00,0.018,0.013768,0.0,0.040215
"""MAC000193""",2014-02-27 02:00:00,0.026,0.032246,0.007773,0.056719
…,…,…,…,…,…
"""MAC000193""",2014-02-27 21:30:00,0.217,0.161677,0.101679,0.221675
"""MAC000193""",2014-02-27 22:00:00,0.169,0.223656,0.16348,0.283831
"""MAC000193""",2014-02-27 22:30:00,0.268,0.206322,0.15287,0.259775


In [35]:
plot_series(data, y_hat, max_insample_length=200, models=["XGBRegressor"], level=[80])

In [ ]:
from functools import partial
from metrics_utils import winkler_score

metrics = [
    mqloss,
    winkler_score,
    coverage,
]
evaluate(
    y_hat,
    metrics=metrics,
    level=[80],
)

unique_id,metric,XGBRegressor
str,str,f64
"""MAC000193""","""mqloss""",0.059518
"""MAC000193""","""winkler_score_level80""",1.190367
"""MAC000193""","""coverage_level80""",0.708333


: 

# Conformal Prediction Intervals vs. Quantile Regression: Pros and Cons

Understanding the strengths and limitations of different uncertainty quantification methods is crucial for effective time series forecasting. Two popular approaches are **conformal prediction intervals** and **quantile regression**. Let’s break down how they compare, using clear explanations and practical examples.

---

## What is Quantile Regression?

Quantile regression directly estimates conditional quantiles of the target variable, such as the 10th, 50th (median), or 90th percentile. Instead of predicting just the mean, it predicts the value below which a certain percentage of observations fall.

- **Example:** Predicting the 90th percentile of electricity demand means estimating a value that 90% of future demands are expected to be below.

---

## What is Conformal Prediction?

Conformal prediction is a wrapper method that can be applied to any forecasting model (including quantile regression!). It uses past prediction errors (residuals) to adjust forecast intervals, providing **distribution-free, finite-sample coverage guarantees**.

---

## Pros and Cons Table

| Feature                        | Conformal Prediction Intervals                | Quantile Regression                       |
|---------------------------------|----------------------------------------------|-------------------------------------------|
| **Assumptions**                | Only exchangeability (very weak)             | Correct model specification for quantiles |
| **Coverage Guarantee**          | Finite-sample, distribution-free             | Only asymptotic, model-dependent          |
| **Model Flexibility**           | Model-agnostic (works with any model)        | Requires models that support quantiles    |
| **Interval Shape**              | Usually symmetric (unless using variants)    | Naturally asymmetric                      |
| **Handles Heteroscedasticity**  | Not by default (unless using local/adaptive) | Yes, if modeled correctly                 |
| **Interpretability**            | Easy to explain and implement                | Directly interprets quantiles             |
| **Computational Cost**          | Requires calibration set and extra steps     | Trains once per quantile                  |
| **Sensitivity to Model Quality**| Honest: intervals widen if model is poor     | Intervals may be misleading if model is misspecified |
| **Works with ML/Deep Models**   | Yes                                         | Yes, if model supports quantiles          |

---

## Pros and Cons Explained

### Conformal Prediction Intervals

**Pros:**
- **Distribution-Free:** No need to assume normality or any specific error distribution.
- **Finite-Sample Guarantees:** The interval will contain the true value with at least the specified probability, even for small datasets.
- **Model-Agnostic:** Can be applied to any forecasting method, including black-box ML models.
- **Honest Uncertainty:** If your model is bad, intervals get wider—reflecting true uncertainty.

**Cons:**
- **Symmetric by Default:** Standard conformal intervals are symmetric around the point forecast, which may not capture asymmetric uncertainty (e.g., skewed errors).
- **Requires Calibration Set:** Needs a separate set of data not used for training.
- **May Not Capture Heteroscedasticity:** Unless using adaptive/local conformal methods, intervals may not adjust for changing variance over time.

---

### Quantile Regression

**Pros:**
- **Directly Models Asymmetry:** Can produce intervals that are wider on one side, matching real-world data distributions.
- **Handles Heteroscedasticity:** Can model changing uncertainty if quantiles are estimated as functions of covariates.
- **Efficient Use of Data:** No need for a separate calibration set.

**Cons:**
- **Model Dependent:** Coverage guarantees only hold if the model is correctly specified and well-calibrated.
- **No Finite-Sample Guarantee:** Coverage is only approximate, especially for small samples or misspecified models.
- **Implementation:** Not all models natively support quantile regression (though many ML libraries do).

---

## When to Use Which?

- **If you want robust, honest intervals with minimal assumptions:**  
    Use **conformal prediction**—especially when using complex or black-box models, or when you’re unsure about the error distribution.

- **If you need asymmetric intervals or want to model changing uncertainty:**  
    Use **quantile regression**—especially when you have enough data and your model can accurately estimate quantiles.

- **Best of Both Worlds:**  
    You can combine both! For example, use quantile regression as your base model, then apply conformal prediction to calibrate the intervals for guaranteed coverage.

---

## Key Takeaway

- **Conformal prediction** is about **guaranteed coverage** and model-agnostic flexibility.
- **Quantile regression** is about **directly modeling the shape** of uncertainty, but relies more on correct model specification.

Both are powerful tools—choose based on your data, goals, and the level of trust you have in your model’s assumptions!